In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv()

In [ ]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

In [ ]:
instructions = "You use your entity tools as a persistent memory top store and recall informaqtion about your conversation"
request = "My name's Sage. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP Protocol"
model = "gemini-3.5-flash"

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent[any](name="agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent[any](name = "agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("converstaion"):
        result = await Runner.run(agent, "My name's Sage. What do you know about me?")
    display_markdown(result.final_output)

In [ ]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY") }}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

In [ ]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize it's outlook. For context, it is {datetime}"
model = "gemini-3.5-flash"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [ ]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent[any](name="agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [1]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    }
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_second=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

NameError: name 'Path' is not defined

In [2]:
INSTRUCTIONS = """You research topics on the web and build up and knowledge base for later.
When you learn something worth keeping, store it and in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it.
"""

model = "gemini-3.5-flash"

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent[any](name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=2)
        display(Markdown(result.final_output))

NameError: name 'MCPServerStdio' is not defined

In [ ]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent[any](name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nividia?")
    display(Markdown(result.final_output))

In [ ]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params ={
        "command": "uvx",
        "args": ["--from ", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key}
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

In [ ]:
instructions = "You answer questions about the stock market."
request = "What was the most recent price that Apple (AAPL) traded at?"
model="gemini-3.5-flash"

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent[any](name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)